In [6]:
import pandas as pd
import numpy as np
import os

branches_df = pd.read_csv('/content/branches_clean.csv')

customers_df = pd.read_csv('/content/customers_clean.csv')

inventory_df = pd.read_csv('/content/inventory_clean.csv')

invoices_df = pd.read_csv('/content/invoices_clean.csv')

payments_df = pd.read_csv('/content/payments_clean.csv')

products_df = pd.read_csv('/content/products_clean.csv')

purchase_orders_header_df = pd.read_csv('/content/purchase_headers_clean.csv')

purchase_orders_lines_df = pd.read_csv('/content/purchase_lines_clean.csv')

sales_orders_header_df = pd.read_csv('/content/sales_headers_clean.csv')

sales_orders_lines_df = pd.read_csv('/content/sales_lines_clean.csv')

stock_ledger_df = pd.read_csv('/content/stock_ledger_clean.csv')

suppliers_df = pd.read_csv('/content/suppliers_clean.csv')

In [7]:
datasets = {
    'branches': branches_df,
    'customers': customers_df,
    'inventory': inventory_df,
    'invoices': invoices_df,
    'payments': payments_df,
    'products': products_df,
    'purchase_headers': purchase_orders_header_df,
    'purchase_lines': purchase_orders_lines_df,
    'sales_headers': sales_orders_header_df,
    'sales_lines': sales_orders_lines_df,
    'stock_ledger': stock_ledger_df,
    'suppliers': suppliers_df
}

### **1. Display all the columns**

In [8]:
for name, df in datasets.items():

    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)

    print(df.columns.tolist())


BRANCHES
['branch_id', 'branch_name', 'city', 'state', 'region', 'warehouse_type', 'warehouse_capacity', 'service_center_available', 'manager_id', 'total_employees', 'avg_monthly_revenue', 'monthly_operational_cost', 'market_demand_index']

CUSTOMERS
['customer_id', 'customer_type', 'industry_segment', 'city', 'state', 'pincode', 'region', 'branch_id', 'credit_limit', 'current_balance', 'payment_terms', 'customer_since', 'last_purchase_date', 'total_purchase_value', 'customer_rating']

INVENTORY
['product_id', 'branch_id', 'opening_stock', 'reorder_level', 'safety_stock', 'max_stock', 'current_stock', 'warehouse_bin']

INVOICES
['invoice_id', 'so_id', 'customer_id', 'branch_id', 'invoice_date', 'due_date', 'total_order_value', 'total_gst_amount', 'grand_total', 'payment_status']

PAYMENTS
['payment_id', 'invoice_id', 'payment_date', 'payment_amount', 'payment_method']

PRODUCTS
['product_id', 'product_name', 'category', 'machine_type', 'brand', 'model_compatibility', 'unit_cost', 'uni

### **2. Find Common Columns**

In [9]:
dataset_names = list(datasets.keys())

common_columns = []

for i in range(len(dataset_names)):

    for j in range(i + 1, len(dataset_names)):

        name1 = dataset_names[i]
        name2 = dataset_names[j]

        columns1 = set(
            datasets[name1].columns
        )

        columns2 = set(
            datasets[name2].columns
        )

        common = sorted(
            columns1.intersection(columns2)
        )

        if common:

            common_columns.append({

                'Dataset 1': name1,

                'Dataset 2': name2,

                'Common Columns': ', '.join(common)

            })

common_columns_df = pd.DataFrame(
    common_columns
)

display(common_columns_df)

,Dataset 1,Dataset 2,Common Columns
0,branches,customers,"branch_id, city, region, state"
1,branches,inventory,branch_id
2,branches,invoices,branch_id
3,branches,purchase_headers,branch_id
4,branches,sales_headers,branch_id
5,branches,stock_ledger,branch_id
6,branches,suppliers,"city, region"
7,customers,inventory,branch_id
8,customers,invoices,"branch_id, customer_id"
9,customers,products,last_purchase_date


### **3. Analyse Potential Keys**

In [10]:
key_analysis = []

for name, df in datasets.items():

    id_columns = [
        column
        for column in df.columns
        if column == 'id'
        or column.endswith('_id')
    ]

    for column in id_columns:

        key_analysis.append({

            'Dataset': name,

            'Column': column,

            'Rows': len(df),

            'Unique Values':
                df[column].nunique(
                    dropna=True
                ),

            'Missing':
                df[column].isna().sum(),

            'Unique':
                df[column].is_unique

        })

key_analysis_df = pd.DataFrame(
    key_analysis
)

display(key_analysis_df)

,Dataset,Column,Rows,Unique Values,Missing,Unique
0,branches,branch_id,6,6,0,True
1,branches,manager_id,6,6,0,True
2,customers,customer_id,500,500,0,True
3,customers,branch_id,500,6,0,False
4,inventory,product_id,180,30,0,False
5,inventory,branch_id,180,6,0,False
6,invoices,invoice_id,18033,17836,0,False
7,invoices,so_id,18033,18033,0,True
8,invoices,customer_id,18033,500,0,False
9,invoices,branch_id,18033,6,0,False


### **4. Check Referential Integrity**

In [11]:
def check_relationship(
    child_df,
    parent_df,
    key,
    child_name,
    parent_name
):

    if key not in child_df.columns:
        print(
            f"{key} not found in {child_name}"
        )
        return None

    if key not in parent_df.columns:
        print(
            f"{key} not found in {parent_name}"
        )
        return None

    unmatched = child_df[
        ~child_df[key].isin(
            parent_df[key].dropna()
        )
    ]

    result = {

        'Child Dataset': child_name,

        'Parent Dataset': parent_name,

        'Key': key,

        'Child Rows': len(child_df),

        'Unmatched Rows': len(unmatched),

        'Unmatched %': round(
            len(unmatched)
            / len(child_df)
            * 100,
            2
        )

    }

    return result

### **5. Automatically Discover Possible Relationships**

In [12]:
relationship_results = []

for i in range(len(dataset_names)):

    for j in range(len(dataset_names)):

        if i == j:
            continue

        child_name = dataset_names[i]
        parent_name = dataset_names[j]

        child_df = datasets[child_name]
        parent_df = datasets[parent_name]

        common_keys = set(
            child_df.columns
        ).intersection(
            parent_df.columns
        )

        for key in common_keys:

            if (
                key == 'id'
                or key.endswith('_id')
            ):

                result = check_relationship(
                    child_df,
                    parent_df,
                    key,
                    child_name,
                    parent_name
                )

                if result:
                    relationship_results.append(
                        result
                    )

relationship_df = pd.DataFrame(
    relationship_results
)

display(relationship_df)

,Child Dataset,Parent Dataset,Key,Child Rows,Unmatched Rows,Unmatched %
0,branches,customers,branch_id,6,0,0.0
1,branches,inventory,branch_id,6,0,0.0
2,branches,invoices,branch_id,6,0,0.0
3,branches,purchase_headers,branch_id,6,0,0.0
4,branches,sales_headers,branch_id,6,0,0.0
...,...,...,...,...,...,...
75,stock_ledger,purchase_headers,branch_id,237230,0,0.0
76,stock_ledger,purchase_lines,product_id,237230,0,0.0
77,stock_ledger,sales_headers,branch_id,237230,0,0.0
78,stock_ledger,sales_lines,product_id,237230,0,0.0


### **6. Examine Sales Relationship**

In [15]:
print("SALES HEADER COLUMNS")
print(
    sales_orders_header_df.columns.tolist()
)

print("\nSALES LINE COLUMNS")
print(
    sales_orders_lines_df.columns.tolist()
)

print("\nPRODUCT COLUMNS")
print(
    products_df.columns.tolist()
)

print("\nCUSTOMER COLUMNS")
print(
    customers_df.columns.tolist()
)

SALES HEADER COLUMNS
['so_id', 'customer_id', 'branch_id', 'order_date', 'delivery_date', 'order_status', 'payment_terms', 'total_order_value', 'total_gst_amount', 'grand_total', 'sales_channel']

SALES LINE COLUMNS
['so_id', 'line_number', 'product_id', 'quantity', 'unit_price', 'gst_rate', 'line_total', 'gst_amount', 'line_grand_total']

PRODUCT COLUMNS
['product_id', 'product_name', 'category', 'machine_type', 'brand', 'model_compatibility', 'unit_cost', 'unit_price', 'margin_percentage', 'gst_rate', 'weight_kg', 'dimensions_cm', 'material_type', 'warranty_months', 'reorder_level', 'safety_stock', 'max_stock_level', 'lead_time_days', 'criticality_level', 'usage_frequency', 'uom', 'last_purchase_price', 'last_purchase_date']

CUSTOMER COLUMNS
['customer_id', 'customer_type', 'industry_segment', 'city', 'state', 'pincode', 'region', 'branch_id', 'credit_limit', 'current_balance', 'payment_terms', 'customer_since', 'last_purchase_date', 'total_purchase_value', 'customer_rating']


### **7. Sales Header → Sales Lines**

In [17]:
sales_integrated_df = sales_orders_lines_df.merge(
    sales_orders_header_df,
    on='so_id',
    how='left',
    suffixes=(
        '_line',
        '_header'
    ),
    indicator=True
)

print(
    "Rows:",
    sales_integrated_df.shape[0]
)

print(
    "\nMerge result:"
)

print(
    sales_integrated_df['_merge']
    .value_counts()
)

# Remove the merge indicator
sales_integrated_df = (
    sales_integrated_df
    .drop(columns=['_merge'])
)

Rows: 130402

Merge result:
_merge
both          130402
left_only          0
right_only         0
Name: count, dtype: int64


### **8. Add Products**

In [18]:
sales_integrated_df = sales_integrated_df.merge(
    products_df,
    on='product_id',
    how='left',
    suffixes=(
        '',
        '_product'
    ),
    indicator=True
)

print(
    sales_integrated_df['_merge']
    .value_counts()
)

sales_integrated_df = (
    sales_integrated_df
    .drop(columns=['_merge'])
)

_merge
both          130402
left_only          0
right_only         0
Name: count, dtype: int64


### **9. Add Customers**

In [19]:
sales_integrated_df = sales_integrated_df.merge(
    customers_df,
    on='customer_id',
    how='left',
    suffixes=(
        '',
        '_customer'
    ),
    indicator=True
)

print(
    sales_integrated_df['_merge']
    .value_counts()
)

sales_integrated_df = (
    sales_integrated_df
    .drop(columns=['_merge'])
)

_merge
both          130402
left_only          0
right_only         0
Name: count, dtype: int64


### **10. Check the Integrated Sales Dataset**

In [20]:
print(
    "Integrated sales shape:",
    sales_integrated_df.shape
)

display(
    sales_integrated_df.head()
)

print("\nMissing values:")

display(
    sales_integrated_df.isna()
    .sum()
    .sort_values(
        ascending=False
    )
    .head(20)
)

Integrated sales shape: (130402, 55)


,so_id,line_number,product_id,quantity,unit_price,gst_rate,line_total,gst_amount,line_grand_total,customer_id,...,pincode,region,branch_id_customer,credit_limit,current_balance,payment_terms_customer,customer_since,last_purchase_date_customer,total_purchase_value,customer_rating
0,SO-990591,1,P018,19,11300,28,214700,60116.0,274816.0,C0070,...,784913,West,AHM001,549454,84153,Net 30,2022-11-10,2024-12-04,4644233,3
1,SO-990591,2,P026,2,310,18,620,111.6,731.6,C0070,...,784913,West,AHM001,549454,84153,Net 30,2022-11-10,2024-12-04,4644233,3
2,SO-990591,3,P026,3,310,18,930,167.4,1097.4,C0070,...,784913,West,AHM001,549454,84153,Net 30,2022-11-10,2024-12-04,4644233,3
3,SO-990591,4,P013,9,1450,18,13050,2349.0,15399.0,C0070,...,784913,West,AHM001,549454,84153,Net 30,2022-11-10,2024-12-04,4644233,3
4,SO-990591,5,P004,3,39800,28,119400,33432.0,152832.0,C0070,...,784913,West,AHM001,549454,84153,Net 30,2022-11-10,2024-12-04,4644233,3



Missing values:


,0
so_id,0
line_number,0
product_id,0
quantity,0
unit_price,0
gst_rate,0
line_total,0
gst_amount,0
line_grand_total,0
customer_id,0


### **11. Purchase Integration**

In [21]:
# First Inspect the Keys
print(
    purchase_orders_header_df.columns.tolist()
)

print(
    purchase_orders_lines_df.columns.tolist()
)

print(
    suppliers_df.columns.tolist()
)

print(
    products_df.columns.tolist()
)

['po_id', 'supplier_id', 'branch_id', 'order_date', 'expected_delivery_date', 'received_date', 'po_status', 'total_cost', 'total_gst_amount', 'grand_total']
['po_id', 'line_number', 'product_id', 'quantity', 'unit_cost', 'gst_rate', 'line_total', 'gst_amount', 'line_grand_total']
['supplier_id', 'supplier_name', 'supplier_type', 'product_category', 'city', 'province', 'region', 'pincode', 'lead_time_days', 'reliability_score', 'import_duty_rate', 'china_tax_id']
['product_id', 'product_name', 'category', 'machine_type', 'brand', 'model_compatibility', 'unit_cost', 'unit_price', 'margin_percentage', 'gst_rate', 'weight_kg', 'dimensions_cm', 'material_type', 'warranty_months', 'reorder_level', 'safety_stock', 'max_stock_level', 'lead_time_days', 'criticality_level', 'usage_frequency', 'uom', 'last_purchase_price', 'last_purchase_date']


In [22]:
purchase_integrated_df = (
    purchase_orders_lines_df.merge(
        purchase_orders_header_df,
        on='po_id',
        how='left',
        suffixes=(
            '_line',
            '_header'
        ),
        indicator=True
    )
)

print(
    purchase_integrated_df['_merge']
    .value_counts()
)

purchase_integrated_df = (
    purchase_integrated_df
    .drop(columns=['_merge'])
)

_merge
both          155495
left_only          0
right_only         0
Name: count, dtype: int64


### **12. Add Supplier Information**

In [23]:
purchase_integrated_df = (
    purchase_integrated_df.merge(
        suppliers_df,
        on='supplier_id',
        how='left',
        suffixes=(
            '',
            '_supplier'
        ),
        indicator=True
    )
)

print(
    purchase_integrated_df['_merge']
    .value_counts()
)

purchase_integrated_df = (
    purchase_integrated_df
    .drop(columns=['_merge'])
)

_merge
both          155495
left_only          0
right_only         0
Name: count, dtype: int64


### **13. Add Product Information**

In [24]:
purchase_integrated_df = (
    purchase_integrated_df.merge(
        products_df,
        on='product_id',
        how='left',
        suffixes=(
            '',
            '_product'
        ),
        indicator=True
    )
)

print(
    purchase_integrated_df['_merge']
    .value_counts()
)

purchase_integrated_df = (
    purchase_integrated_df
    .drop(columns=['_merge'])
)

_merge
both          155495
left_only          0
right_only         0
Name: count, dtype: int64


### **Save Integrated Datasets**

In [25]:
os.makedirs(
    '/content/week-02-integration',
    exist_ok=True
)

sales_integrated_df.to_csv(
    '/content/week-02-integration/'
    'sales_integrated.csv',
    index=False
)

purchase_integrated_df.to_csv(
    '/content/week-02-integration/'
    'purchase_integrated.csv',
    index=False
)

common_columns_df.to_csv(
    '/content/week-02-integration/'
    'common_columns.csv',
    index=False
)

key_analysis_df.to_csv(
    '/content/week-02-integration/'
    'key_analysis.csv',
    index=False
)

relationship_df.to_csv(
    '/content/week-02-integration/'
    'relationship_analysis.csv',
    index=False
)

print("Integration files saved.")

Integration files saved.
